# Bubble Bi

### Teaching a machine to read the market like a language

---

**This notebook is the project.** You run it from top to bottom. Each section explains what is about to happen in plain language, then does it, then shows you the result and what it means.

You do not need to read any code to follow along. The heavy machinery lives in a folder called `backup/`, out of sight. This notebook calls it and explains it.

*Right now this notebook only introduces the project. The working sections come next.*

## 1. The problem

Every trading day, a stock leaves behind a small trace: where it opened, how high and low it went, where it closed, and how many shares changed hands. Six numbers. Do that for 30 companies over 16 years and you have about **125,000 of these daily traces**.

Hidden somewhere in that pile is structure. Markets are not pure noise — they have moods. There are calm grinding-upward stretches, panicky sell-offs, quiet drifting periods where nothing happens, sharp reversals. A human trader who has watched screens for twenty years recognises these on sight, but struggles to say precisely what they are looking at.

**The question this project asks:** can a machine learn to recognise those recurring market states on its own — nobody labels them, nobody defines them in advance — and then use that vocabulary to say something useful about what happens next?

## 2. The core idea: give the market a language

Here is the whole project in one analogy.

Large language models like ChatGPT work by breaking text into **tokens** — roughly, words — and then learning to predict the next one. Feed it *"the cat sat on the"* and it answers *"mat"*. It learned that by reading enormous amounts of text and noticing which words tend to follow which.

**We do the same thing to the stock market.**

Instead of words, our tokens are *market states*. We compress everything that happened to one stock on one day — its price action, its volatility, its trading friction, its memory of the past — down into **a single token drawn from a fixed vocabulary of 512 possible states**.

So Apple on 3 March becomes token #147. Apple on 4 March becomes token #147 again — it was a similar kind of day. Then something breaks, and 5 March becomes token #391.

Once every stock-day is a token, a stock's history is literally **a sentence**:

```
AAPL:   ... 147  147  147  391  208  208  63 ...  →  what comes next?
MSFT:   ... 12   12   88   88   88  147 147 ...   →  what comes next?
```

And now we can do to the market what GPT does to text: **read the sentence, predict the next token.**

## 3. Why bother turning numbers into words?

A fair objection: the data is already numbers. Why squeeze it into a vocabulary of 512 states instead of just feeding the raw numbers to a model?

Because **the squeezing is the point.**

If you force a model to describe every possible market day using only 512 words, it cannot memorise. It has to decide what actually matters and throw the rest away. To do that well it must discover that *these* days are fundamentally the same kind of day, and *those* days are a different kind. That forced economy is what makes it find real, recurring structure instead of latching onto noise.

It also gives us something rare in finance: **an interpretable handle.** We can go back and ask *"what does token #391 actually look like?"* and get an answer — high volatility, wide spreads, heavy volume, breaking downward. The vocabulary becomes a description of how the market behaves.

The technique that does this squeezing is called a **VQ-VAE** (Vector Quantised Variational Autoencoder). Two facts are all you need about it:

- It learns by **reconstruction**: it compresses a day down to one token, then tries to rebuild the original day from that token alone. If it can rebuild it, the token captured what mattered.
- Nobody tells it what the 512 states should be. **It invents them.**

## 4. The three stages

```
        ┌─────────────────────────────────────────────────────────┐
        │  RAW DATA   30 stocks × 4,153 days × 22 measurements     │
        └────────────────────────────┬────────────────────────────┘
                                     │
        ┌────────────────────────────▼────────────────────────────┐
        │  STAGE 1 — THE TOKENIZER                     ✅ built    │
        │                                                          │
        │  Compresses each stock-day into ONE token out of 512.    │
        │  Looks at the stock's own recent history AND at what     │
        │  the whole market was doing that day.                    │
        │                                                          │
        │  Output:  a grid of tokens — the market, as sentences.   │
        └────────────────────────────┬────────────────────────────┘
                                     │
        ┌────────────────────────────▼────────────────────────────┐
        │  STAGE 2 — THE PREDICTOR                     ✅ built    │
        │                                                          │
        │  A GPT-style model (the same architecture family as      │
        │  Llama 3) that reads a stock's token sentence and        │
        │  predicts the next token.                                │
        │                                                          │
        │  Output:  a model that understands market grammar.       │
        └────────────────────────────┬────────────────────────────┘
                                     │
        ┌────────────────────────────▼────────────────────────────┐
        │  STAGE 3 — THE TRADER                    ⬜ not built    │
        │                                                          │
        │  We cut the prediction head off the model and keep its   │
        │  "understanding". A reinforcement-learning agent uses     │
        │  that understanding to act: go long, go short, stay      │
        │  flat, set stops.                                        │
        └──────────────────────────────────────────────────────────┘
```

Each stage is trained separately and then **frozen** before the next one is built on top of it. That is deliberate: it means when something works, we know *which* part is working.

## 5. Where the idea comes from — and where we deliberately break from it

The project is inspired by a research paper, **STORM** *(Zhao et al., WSDM '26)* — the PDF sits in this folder. STORM introduced the idea of using this kind of dual compression on financial data: one view of a single stock over time, another view of the whole market on a single day.

**We take their tokenizer idea and then go somewhere else with it.**

| | STORM (the paper) | Bubble Bi (this project) |
|---|---|---|
| What the tokens feed | A factor model that predicts returns directly | **A separate GPT-style Transformer** |
| How it predicts | A linear return head | **Next-token prediction, like a language model** |
| What comes after | A trading agent bolted on | **The Transformer's "understanding" becomes the agent's eyes** |

The bet we are making: **treating the market as a language, and learning its grammar, is a better foundation for a trading agent than predicting a number.** A model that knows what usually follows a panic is more useful than one that outputs "+0.3%".

This is a genuine deviation, not an implementation detail. It is the interesting claim of the project, and it is the thing that could be wrong.

## 6. What the machine actually sees

For every stock, on every day, we compute **22 measurements**. All of them are strictly *backward-looking* — computed only from what was already known that day. (This matters enormously; see the warning below.)

They fall into five families:

| Family | What it captures | Examples |
|---|---|---|
| **Price action** | Where the price is going | returns, moving-average ratios, RSI, MACD |
| **Instability** | How violent the movement is | Parkinson, Garman-Klass, Yang-Zhang volatility |
| **Memory** | Whether the past still echoes | Hurst exponent, fractionally-differenced price, entropy |
| **Friction** | The hidden cost of trading it | Amihud illiquidity, Roll spread, Corwin-Schultz |
| **Flow** | Who is pushing, and how hard | volume dynamics, on-balance volume |

That is **30 stocks × 4,153 days × 22 numbers ≈ 2.7 million measurements**, and the tokenizer's job is to boil each stock-day down to a single number between 0 and 511.

---

> ### ⚠️ The one rule that cannot be broken
>
> **Nothing may ever look into the future.**
>
> If a single one of those 22 measurements accidentally peeks even one day ahead, the model will appear to make brilliant predictions — and every one of them will be a lie. It will look like it works, and it will lose real money.
>
> This is the most common way financial machine-learning projects fool themselves. The project defends against it with an automated test that takes the data, deletes the future, recomputes everything, and checks that **not a single past value changed**. That test runs over all 22 measurements, every time.

## 7. Honest status

Some things work. Some numbers are humbling. Both are reported here, because a project that only reports its wins is not a project you can trust.

**What is built and working**

- The full data pipeline, with the no-lookahead rule enforced by automated tests.
- The tokenizer. It uses **80% of its 512-word vocabulary** — meaning it genuinely discovered a rich set of market states rather than collapsing into a handful. (Getting this to work was the hardest part of the project; an earlier design collapsed to using only 13%.)
- The predictor. On the previous version of the data it reached **58.6% accuracy** at guessing the next token out of 512 possibilities.

**What is honest to admit**

- **Predicting markets is brutally hard.** The standard yardstick here is *RankIC*, which measures how well predicted rankings match actual rankings. A simple linear model on our data scores about **0.006**. That is a tiny number — and it is *normal*. Real quantitative funds live and die in this range. Nobody should read "58.6% next-token accuracy" and imagine easy money.
- **The tokenizer and predictor numbers above are now stale.** We recently expanded from 10 measurements per day to 22, which means everything must be retrained before those figures can be quoted again.
- **Stage 3 — the actual trader — does not exist yet.** No backtest, no costs, no slippage, no proof of profitability. Anyone claiming otherwise about this project is wrong.

**What this notebook will not do:** tell you it found a money machine.

## 8. What happens next in this notebook

The sections below this one are where the project actually runs. Each one follows the same shape: *here is what we are about to do → here is why → run it → here is what the result means.*

| Section | What you will do |
|---|---|
| **Setup** | Get the environment ready. One cell. |
| **Get the data** | Download 16 years of daily prices for 30 companies. |
| **Build the measurements** | Turn raw prices into the 22 measurements — and *prove* none of them peek at the future. |
| **Set the bar** | Run a deliberately simple model first, so we have an honest score to beat. |
| **Train the tokenizer** | Watch the machine invent its 512-word vocabulary of market states. |
| **Look at the vocabulary** | Ask what the words actually *mean*. |
| **Train the predictor** | Teach a GPT-style model to read market sentences. |
| **Judge it** | Did it beat the simple model? Be honest. |

---

*The frozen, fully-tested implementation of everything above lives in `backup/`. This notebook is the guided tour; that folder is the engine room.*

**Nothing runs yet. Continue to the next section when it is built.**